# 01 — データ探索・EDA・検証

- 保存済みデータの品質・分布・分割を確認する。
- 処理は `src/`、Notebookは分析・可視化を担当。
- 推薦の挙動は02、性能比較は03。

## 1. Overview

| 対象 | 内容・規模 |
| --- | --- |
| PostgreSQL | Music4All-Onion v2。公開済みsnapshotを指定（初回100曲・100ユーザー） |
| 既定モード | 合成fixture：4曲・2ユーザー。実Music4Allではない |
| 参考資料 | 全量イベントの保存済み集計。snapshotとは別の対象 |
| 出典 | [Zenodo record 15394646](https://zenodo.org/records/15394646) |

- 現snapshotに再生関係・ユーザー属性はなく、正式なTrain / Validation / Testは未作成。
- 起動：`uv run --locked --group data --group notebook --group tracking jupyter lab`
- DBを読む場合は起動時に `.env` を渡す。接続値はセルに書かない。
- 既定実行はDB・配布ファイル・常駐MLflow不要。

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path
from uuid import UUID

import matplotlib.pyplot as plt
from IPython.display import display
from pandas import DataFrame

from music_recsys.analysis import split_candidate_tables, temporal_rows
from music_recsys.snapshot_analysis import (
    load_fixture,
    load_snapshot_frames,
    snapshot_eda,
    validation_rows,
)

ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "configs/toy.toml").is_file()
)
SNAPSHOT_ID = None  # 公開済みsnapshotのUUID文字列。NoneならDBに接続しない。
DATASET_KEY = "music4all-onion"
VERSION_LABEL = "v2"
FIXTURE_PATH = ROOT / "examples/catalog/snapshot.json"
PROFILE_PATH = Path(
    os.environ.get("MUSIC_RECSYS_PROFILE", str(ROOT / "experiments/music4all/eda/profile.json"))
)

mode = "PostgreSQL saved snapshot" if SNAPSHOT_ID else "Saved synthetic fixture (not the database)"
display(
    {
        "mode": mode,
        "dataset": DATASET_KEY if SNAPSHOT_ID else "synthetic-catalog",
        "version": VERSION_LABEL if SNAPSHOT_ID else "fixture-v1",
        "snapshot_id": SNAPSHOT_ID,
        "source": "https://zenodo.org/records/15394646"
        if SNAPSHOT_ID
        else "examples/catalog/snapshot.json",
        "expected_tracks": 100 if SNAPSHOT_ID else 4,
        "expected_users": 100 if SNAPSHOT_ID else 2,
    }
)

## 2. Data Loading

| モード | 読込み |
| --- | --- |
| PostgreSQL | 指定した公開済みsnapshotのSQL結果をDataFrameへ |
| fixture | 保存済み合成データをDataFrameへ |
| 全量イベントの参考資料 | 保存済み集計JSONのみ |

- SQLは読み取り専用。版・snapshotを固定し、Parquetも照合する。
- 表示は規模・カラム・出典。取り込み・分割処理は実行しない。

In [ ]:
if SNAPSHOT_ID:
    from music_recsys.ingestion.postgres import engine_from_environment

    engine = engine_from_environment()
    try:
        snapshot = load_snapshot_frames(
            engine,
            ROOT / "data/imports",
            UUID(SNAPSHOT_ID),
            dataset_key=DATASET_KEY,
            version_label=VERSION_LABEL,
        )
    finally:
        engine.dispose()
else:
    snapshot = load_fixture(FIXTURE_PATH)

tracks_df, users_df = snapshot.tracks, snapshot.users
profile = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))
display(snapshot.overview)
display(
    {
        "tracks_frame": {"rows": len(tracks_df), "columns": list(tracks_df.columns)},
        "users_frame": {"rows": len(users_df), "columns": list(users_df.columns)},
        "reference_eda": {
            key: profile[key] for key in ("n_events", "n_users", "n_items", "min_time", "max_time")
        },
    }
)

## 3. Data Validation

| 確認 | 内容 |
| --- | --- |
| 件数・ID | 元ID／UUIDの欠損・重複・形式 |
| タグ | `payload=NULL` は未取得、`tags={}` は取得済み・タグなし |
| 実DB | 所属・参照・作成run・公開状態・manifest／fingerprint |
| Parquet | hash・schema・全行内容 |

- DB検証失敗は読込み時にエラー。
- fixtureのDB／Parquet検証は `not checked`。
- `expected=None` の欠損タグ数・空タグ数は観測値。存在だけではエラーにしない。

In [ ]:
validation = validation_rows(
    snapshot,
    expected_tracks=100 if SNAPSHOT_ID else 4,
    expected_users=100 if SNAPSHOT_ID else 2,
)
display(DataFrame(validation))

## 4. EDA

| 図 | 集計対象・定義 |
| --- | --- |
| タグの有無・出現率 | 読み込んだ全曲が分母 |
| 曲ごとのタグ数 | 取得済み曲。空タグは0 |
| 重み分布 | 配布値1〜100。嗜好確率・推薦scoreではない |

- この章を主な探索に使う。fixtureの図は合成例。
- ユーザーはIDのみ。年齢・地域・嗜好などの属性分布は未分析。

In [ ]:
eda = snapshot_eda(snapshot)
display(eda["tag_coverage"])
display(eda["tags"].head(15))
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
coverage = eda["tag_coverage"]
axes[0, 0].bar(coverage["status"], coverage["n_tracks"], color=["#8b507b", "#c77936", "#256b78"])
axes[0, 0].set(title="Tag availability", ylabel="Tracks")
top_tags = eda["tags"].head(10)
axes[0, 1].barh(top_tags["tag"], top_tags["track_fraction"], color="#256b78")
axes[0, 1].invert_yaxis()
axes[0, 1].set(title="Tag occurrence among all tracks", xlabel="Track fraction", xlim=(0, 1))
axes[1, 0].hist(eda["tag_counts"], bins="auto", color="#256b78", edgecolor="white")
axes[1, 0].set(title="Tags per track (known metadata)", xlabel="Tag count", ylabel="Tracks")
axes[1, 1].hist(eda["tag_weights"], bins=10, range=(1, 101), color="#c77936", edgecolor="white")
axes[1, 1].set(title="Supplied tag weights", xlabel="Weight", ylabel="Tag assignments")
fig.suptitle(f"Catalog EDA: {snapshot.overview['dataset_key']} / {snapshot.mode}")
fig.tight_layout()
plt.show()

### 参考：全量イベントの時刻分布

- 保存済み全量集計。カタログsnapshotとは対象が異なる。
- 配布時刻のタイムゾーンは未確認。UTC変換なし。
- 1970年の1件は元LFM-2bの記載期間外。図は2005-02-14以降に拡大し、集計には残す。
- 別のJSONを指定した場合は、Data Loadingの表で規模を確認する。

In [ ]:
temporal = temporal_rows(profile)
days = [row["day"] for row in temporal]
counts = [row["n_events"] for row in temporal]
cumulative = [row["cumulative_fraction"] for row in temporal]
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(days, counts, color="#256b78", linewidth=0.8)
axes[0].set(title="Reference only: full-event temporal distribution", ylabel="Events per day")
axes[1].plot(days, cumulative, color="#256b78")
axes[1].set(ylabel="Cumulative event fraction", ylim=(0, 1.02))
plot_start = max(days[0], datetime(2005, 2, 14))
if days[-1] > plot_start:
    axes[1].set_xlim(plot_start, days[-1])
fig.autofmt_xdate()
fig.tight_layout()
plt.show()
display(profile["train"])

## 5. Data Splitting

| 正式な保存済み分割 | 状態 |
| --- | --- |
| Train / Validation / Test | 未作成 |
| 件数・時系列・cold-start率・ground truth件数 | 未検証 |

- 現snapshotに再生関係はない。分割の作成は行わず、保存物ができた段階で検証する。

### 参考：保存済み分割候補

- 全量EDAの候補境界・集計値。**正式分割・採用済み比率ではない。**

| 候補 | 学習側 |
| --- | --- |
| `validation` / `test_train_only` | train |
| `test_refit_train_validation` | train + validation |

| 集計 | 定義 |
| --- | --- |
| cold-start率 | 各期間のdistinct user-track pairsが分母 |
| 分類 | `cold_user → cold_item → seen → eligible` の優先順で排他的に分類 |
| cold_item率 | cold_userに分類されなかったcold itemペアの割合 |
| `novel_truth_pairs` | 適格な未聴正解ペア数 |
| `users_with_novel_truth` | 未聴正解を持つユーザー数 |

- 両方coldならcold_userに分類。期間内ペアがなければ率は未定義（0にしない）。
- 件数・日付境界・未聴正解数の確認用。推薦精度ではない。

In [ ]:
display(
    {"status": "Reference candidate only; no official split artifacts", **profile["boundaries"]}
)
candidates = split_candidate_tables(profile)
display(DataFrame(candidates["events"]))
display(DataFrame(candidates["eligibility"]))
phases = candidates["eligibility"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = [row["condition"] for row in phases]
positions = list(range(len(phases)))
for metric, offset, color, label in (
    ("cold_user_fraction", -0.18, "#8b507b", "Cold user (exclusive)"),
    ("cold_item_fraction", 0.18, "#c77936", "Cold item (exclusive)"),
):
    defined = [
        (position, row[metric]) for position, row in enumerate(phases) if row[metric] is not None
    ]
    axes[0].bar(
        [position + offset for position, _ in defined],
        [value for _, value in defined],
        width=0.36,
        color=color,
        label=label,
    )
undefined = [row["condition"] for row in phases if row["unique_pairs"] == 0]
if undefined:
    axes[0].text(
        0.02,
        0.97,
        "Undefined (no pairs): " + ", ".join(undefined),
        transform=axes[0].transAxes,
        va="top",
        fontsize=8,
    )
axes[0].set(
    title="Reference: cold-start pair fractions",
    ylabel="Fraction of unique pairs",
    xticks=positions,
    xticklabels=labels,
    ylim=(0, 1),
)
axes[0].legend()
axes[1].bar(labels, [row["novel_truth_pairs"] for row in phases], color="#256b78")
axes[1].set(title="Reference: eligible novel truth", ylabel="Unique user-track pairs")
for ax in axes:
    ax.tick_params(axis="x", rotation=15, labelsize=8)
fig.tight_layout()
plt.show()

## 6. Summary

- 出典・版・snapshot・規模、件数・欠損・ID・タグを確認。
- EDAは曲タグが中心。再生履歴・ユーザー属性は現snapshotにない。
- 全量イベントと分割候補は参考資料。正式分割は未検証。
- 推薦の挙動は02、モデルの性能比較は03で確認。